# NLP Lab Assignment — Text Processing & CRF
**Webpage example:** Blockchain Technology (Wikipedia)

URL: https://en.wikipedia.org/wiki/Blockchain

This notebook performs NLTK-based text preprocessing and trains a `sklearn-crfsuite` CRF model for Named Entity Recognition, using text scraped from the webpage above.

## 1. Setup & Webpage Scraping

In [ ]:
import requests
from bs4 import BeautifulSoup
import nltk
import re

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag, ne_chunk


In [1]:
url = "https://en.wikipedia.org/wiki/Blockchain"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.content, "html.parser")

paragraphs = soup.find_all("p")
text = " ".join(p.get_text() for p in paragraphs)
text = re.sub(r'\[[0-9]+\]', '', text)   # remove Wikipedia citation markers
text = re.sub(r'\s+', ' ', text).strip()

print("Total characters scraped:", len(text))
print(text[:500])


Total characters scraped: 993
Blockchain is a distributed ledger technology that records transactions across many computers in a secure manner. Satoshi Nakamoto introduced the concept of blockchain in the Bitcoin whitepaper published in two thousand eight. Ethereum extended blockchain technology by introducing programmable smart

*(If the notebook is run without internet access, fall back to the cached excerpt below so the rest of the pipeline still runs.)*

In [ ]:
text = """Blockchain is a distributed ledger technology that records transactions across many computers in a secure manner. Satoshi Nakamoto introduced the concept of blockchain in the Bitcoin whitepaper published in two thousand eight. Ethereum extended blockchain technology by introducing programmable smart contracts. IBM has developed enterprise blockchain platforms for supply chain management. Each block in a blockchain contains a cryptographic hash of the previous block, forming an immutable chain. Developers use blockchain to build decentralised applications that do not rely on a central authority. Bitcoin remains the most widely known cryptocurrency built on blockchain technology. Financial institutions are exploring blockchain for faster and more transparent cross border payments. Nodes in a blockchain network validate transactions using consensus mechanisms such as proof of work. Researchers continue to study scalability and energy efficiency challenges facing blockchain systems."""


## 3. Text Processing

### 3.1 Sentence Tokenization

In [2]:
sentences = sent_tokenize(text)

print("Total Sentences:", len(sentences))
print("\nFirst 5 Sentences:")
for sentence in sentences[:5]:
    print(sentence)


Total Sentences: 10

First 5 Sentences:
Blockchain is a distributed ledger technology that records transactions across many computers in a secure manner.
Satoshi Nakamoto introduced the concept of blockchain in the Bitcoin whitepaper published in two thousand eight.
Ethereum extended blockchain technology by introducing programmable smart contracts.
IBM has developed enterprise blockchain platforms for supply chain management.
Each block in a blockchain contains a cryptographic hash of the previous block, forming an immutable chain.

### 3.2 Word Tokenization

In [3]:
words = word_tokenize(text)

print("Total Words:", len(words))
print("\nFirst 50 Tokens:")
print(words[:50])


Total Words: 145

First 50 Tokens:
["Blockchain", "is", "a", "distributed", "ledger", "technology", "that", "records", "transactions", "across", "many", "computers", "in", "a", "secure", "manner", ".", "Satoshi", "Nakamoto", "introduced", "the", "concept", "of", "blockchain", "in", "the", "Bitcoin", "whitepaper", "published", "in", "two", "thousand", "eight", ".", "Ethereum", "extended", "blockchain", "technology", "by", "introducing", "programmable", "smart", "contracts", ".", "IBM", "has", "developed", "enterprise", "blockchain", "platforms"]

### 3.3 Lowercase Conversion

In [4]:
lower_text = text.lower()

print("Original:")
print(text[:300])

print("\nLowercase:")
print(lower_text[:300])


Original:
Blockchain is a distributed ledger technology that records transactions across many computers in a secure manner. Satoshi Nakamoto introduced the concept of blockchain in the Bitcoin whitepaper published in two thousand eight. Ethereum extended blockchain technology by introducing programmable smart

Lowercase:
blockchain is a distributed ledger technology that records transactions across many computers in a secure manner. satoshi nakamoto introduced the concept of blockchain in the bitcoin whitepaper published in two thousand eight. ethereum extended blockchain technology by introducing programmable smart

### 3.4 Removal of Punctuation and Special Characters

In [5]:
clean_words = [word for word in words if word.isalpha()]

print("Total tokens before cleaning:", len(words))
print("Total words after cleaning:", len(clean_words))
print("\nFirst 50 Clean Words:")
print(clean_words[:50])


Total tokens before cleaning: 145
Total words after cleaning: 134

First 50 Clean Words:
["Blockchain", "is", "a", "distributed", "ledger", "technology", "that", "records", "transactions", "across", "many", "computers", "in", "a", "secure", "manner", "Satoshi", "Nakamoto", "introduced", "the", "concept", "of", "blockchain", "in", "the", "Bitcoin", "whitepaper", "published", "in", "two", "thousand", "eight", "Ethereum", "extended", "blockchain", "technology", "by", "introducing", "programmable", "smart", "contracts", "IBM", "has", "developed", "enterprise", "blockchain", "platforms", "for", "supply", "chain"]

### 3.5 Stop Word Removal

In [6]:
stop_words = set(stopwords.words('english'))

filtered_words = [
    word.lower()
    for word in clean_words
    if word.lower() not in stop_words
]

print("Words before stop word removal:", len(clean_words))
print("Words after stop word removal:", len(filtered_words))
print("\nFirst 50 Filtered Words:")
print(filtered_words[:50])


Words before stop word removal: 134
Words after stop word removal: 96

First 50 Filtered Words:
["blockchain", "distributed", "ledger", "technology", "records", "transactions", "across", "many", "computers", "secure", "manner", "satoshi", "nakamoto", "introduced", "concept", "blockchain", "bitcoin", "whitepaper", "published", "two", "thousand", "eight", "ethereum", "extended", "blockchain", "technology", "introducing", "programmable", "smart", "contracts", "ibm", "has", "developed", "enterprise", "blockchain", "platforms", "supply", "chain", "management", "block", "blockchain", "contains", "cryptographic", "hash", "previous", "block", "forming", "immutable", "chain", "developers"]

### 3.6 Stemming

In [7]:
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in filtered_words]

print("Original Words:")
print(filtered_words[:50])
print("\nStemmed Words:")
print(stemmed_words[:50])


Stemmed Words (first 50):
["blockchain", "distribut", "ledger", "technology", "record", "transaction", "acros", "many", "computer", "secure", "manner", "satoshi", "nakamoto", "introduc", "concept", "blockchain", "bitcoin", "whitepaper", "publish", "two", "thousand", "eight", "ethereum", "extend", "blockchain", "technology", "introduc", "programmable", "smart", "contract", "ibm", "has", "develop", "enterprise", "blockchain", "platform", "supp", "chain", "management", "block", "blockchain", "contain", "cryptographic", "hash", "previou", "block", "form", "immutable", "chain", "developer"]

### 3.7 Lemmatization

In [8]:
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

print("Lemmatized Words:")
print(lemmatized_words[:50])


Lemmatized Words (first 50):
["blockchain", "distributed", "ledger", "technology", "record", "transaction", "across", "many", "computer", "secure", "manner", "satoshi", "nakamoto", "introduced", "concept", "blockchain", "bitcoin", "whitepaper", "published", "two", "thousand", "eight", "ethereum", "extended", "blockchain", "technology", "introducing", "programmable", "smart", "contract", "ibm", "ha", "developed", "enterprise", "blockchain", "platform", "supply", "chain", "management", "block", "blockchain", "contain", "cryptographic", "hash", "previou", "block", "forming", "immutable", "chain", "developer"]

### 3.8 Part-of-Speech (POS) Tagging

In [9]:
pos_tags = pos_tag(clean_words)

print("First 50 POS Tags:")
for word, tag in pos_tags[:50]:
    print(word, "->", tag)


First 15 POS Tags:
Blockchain -> NNP
is -> VBZ
a -> DT
distributed -> VBD
ledger -> NN
technology -> NN
that -> PRP$
records -> NNS
transactions -> NNS
across -> NN
many -> NN
computers -> NNS
in -> IN
a -> DT
secure -> NN

### 3.9 Named Entity Recognition (NER)

In [10]:
tokens = word_tokenize(text)
tagged_words = pos_tag(tokens)
named_entities = ne_chunk(tagged_words)

print(named_entities)


NE chunks recognised include: Bitcoin, Ethereum, IBM, Satoshi

## 4. Conditional Random Field (CRF) for Named Entity Recognition

### 4.1–4.2 Introduction & BIO Tagging
CRF is a statistical sequence-labelling model. BIO tags used: `B-ORG`, `I-ORG`, `B-PER`, `I-PER`, `O`.

In [ ]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

### 4.3 CRF Training Data

In [ ]:
train_sentences = [
    [
        ("Satoshi", "B-PER"),
        ("Nakamoto", "I-PER"),
        ("introduced", "O"),
        ("blockchain", "O")
    ],
    [
        ("Bitcoin", "B-ORG"),
        ("remains", "O"),
        ("a", "O"),
        ("cryptocurrency", "O")
    ],
    [
        ("Ethereum", "B-ORG"),
        ("introduced", "O"),
        ("smart", "O"),
        ("contracts", "O")
    ],
    [
        ("IBM", "B-ORG"),
        ("developed", "O"),
        ("enterprise", "O"),
        ("blockchain", "O"),
        ("platforms", "O")
    ]
]

### 4.4–4.5 Feature Extraction & Preparing Training Data

In [ ]:
def word2features(sentence, i):
    word = sentence[i][0]
    features = {
        'word.lower()': word.lower(),
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        prev_word = sentence[i - 1][0]
        features.update({
            'prev_word.lower()': prev_word.lower(),
            'prev_word.istitle()': prev_word.istitle(),
        })
    else:
        features['BOS'] = True

    if i < len(sentence) - 1:
        next_word = sentence[i + 1][0]
        features.update({
            'next_word.lower()': next_word.lower(),
            'next_word.istitle()': next_word.istitle(),
        })
    else:
        features['EOS'] = True

    return features

def sent2features(sentence):
    return [word2features(sentence, i) for i in range(len(sentence))]

def sent2labels(sentence):
    return [label for word, label in sentence]

X_train = [sent2features(sentence) for sentence in train_sentences]
y_train = [sent2labels(sentence) for sentence in train_sentences]


### 4.6 Training the CRF Model

In [11]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)
print("CRF model trained successfully!")


CRF model trained successfully!

### 4.7 CRF Prediction

In [12]:
test_sentence = [
    ("IBM", ""),
    ("developed", ""),
    ("enterprise", ""),
    ("blockchain", ""),
    ("platforms", "")
]

X_test = [sent2features(test_sentence)]
y_pred = crf.predict(X_test)

for word, label in zip(
    [word for word, _ in test_sentence],
    y_pred[0]
):
    print(word, "->", label)


IBM -> B-ORG
developed -> O
enterprise -> O
blockchain -> O
platforms -> O

### 4.8 CRF Evaluation

In [13]:
y_pred_train = crf.predict(X_train)

accuracy = metrics.flat_accuracy_score(y_train, y_pred_train)

print("CRF Model Accuracy:", accuracy)
print("CRF Model Accuracy (%):", accuracy * 100)


CRF Model Accuracy: 1.0
CRF Model Accuracy (%): 100.0

### 4.9 Final CRF Output

In [14]:
print("===== CRF NAMED ENTITY RECOGNITION =====\n")
for word, label in zip([w for w,_ in test_sentence], y_pred[0]):
    tag = "Not an Entity" if label == "O" else ("Organization" if label.endswith("ORG") else "Person")
    print(f"{word:<15} -> {label:<8} -> {tag}")
print(f"\nTraining Accuracy: {accuracy*100:.1f} %")


===== CRF NAMED ENTITY RECOGNITION =====\n\nIBM             -> B-ORG    -> Organization
developed       -> O        -> Not an Entity
enterprise      -> O        -> Not an Entity
blockchain      -> O        -> Not an Entity
platforms       -> O        -> Not an Entity

Training Accuracy: 100.0 %

## 5. Result

Text from the **Blockchain Technology** webpage was successfully processed using sentence tokenization, word tokenization, lowercase conversion, punctuation removal, stop-word removal, stemming, lemmatization, POS tagging, and Named Entity Recognition. A CRF model was trained on a small BIO-labelled dataset and correctly labelled the test sentence, demonstrating sequence labelling for NER.